# STOCKSENSE: Feature Engineering & Machine Learning Pipeline
**NovaMart Retail Use Case (IntelliData 2026 Hackathon)**

This notebook provides a complete narrative walkthrough of the **STOCKSENSE** feature engineering, target construction, chronological model benchmarking, probability calibration, and replenishment decision intelligence pipeline.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add root directory to sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT_DIR))

from src.config import load_config
from src.validation import DataValidator
from src.synthetic_data import generate_synthetic_master_table
from src.features.builder import FeatureBuilder
from src.targets import build_targets
from src.evaluation import chronological_split
from src.models.regression import DemandForecasterSuite
from src.models.classification import StockoutClassifierSuite
from src.recommend import RecommendationEngine

## 1. Data Loading & Schema Validation
We load the master table (or synthetic demo table) and run schema verification, grain uniqueness checks, and date continuity validation.

In [ ]:
config = load_config()
df_raw = generate_synthetic_master_table(num_days=120)
validator = DataValidator(config)
df_clean, summary = validator.validate(df_raw)
print("Validation Summary:", summary)

## 2. Zero-Leakage Feature Engineering
We compute 7 modular feature groups: Time, Lags, Shifted Rolling Windows, Inventory Ratios, Price & Promo, External Weather, and Sparse History Fallback.

In [ ]:
fb = FeatureBuilder()
df_feat = fb.transform(df_clean)
print("Engineered Feature DataFrame Shape:", df_feat.shape)

## 3. Target Engineering (Regression & Classification)
- `next_7_day_demand`: Sum of units_sold over t+1..t+7.
- `stockout_flag`: Operational stock-out definition (stock <= 1 day demand OR demand > available stock).

In [ ]:
df_target = build_targets(df_feat)
print("Target DataFrame Shape:", df_target.shape)

## 4. Chronological Train / Val / Test Split

In [ ]:
tr_df, val_df, ts_df = chronological_split(df_target)
print(f"Train: {len(tr_df)}, Val: {len(val_df)}, Test: {len(ts_df)}")

## 5. Model 1 Benchmarking: 7-Day Demand Forecast (Regression)

In [ ]:
reg_suite = DemandForecasterSuite()
reg_bench, reg_info = reg_suite.benchmark_and_train(tr_df, val_df, ts_df)
display(reg_bench)

## 6. Model 2 Benchmarking: Stock-Out Risk (Classification & Calibration)

In [ ]:
clf_suite = StockoutClassifierSuite()
clf_bench, clf_info = clf_suite.benchmark_and_train(tr_df, val_df, ts_df)
display(clf_bench)

## 7. Decision Intelligence & Recommendation Engine

In [ ]:
engine = RecommendationEngine()
latest_snapshot = ts_df[ts_df["date"] == ts_df["date"].max()].copy()
latest_snapshot["pred_7_day_demand"] = reg_suite.best_pipeline.predict(latest_snapshot)
latest_snapshot["pred_stockout_prob"] = clf_suite.best_calibrated_pipeline.predict_proba(latest_snapshot)[:, 1]

rec_table = engine.calculate_recommendations(latest_snapshot)
display(rec_table)